# 01 · Prepare DEAM embeddings (C4)

**Goal:** turn the DEAM music dataset into training data for my emotion head.

For every song in DEAM:
1. Cut the **annotated part** of the audio into **10 s windows with a 5 s hop**, exactly like the backend does for YouTube audio (`backend/audio_utils.py`).
2. Run each window through the pretrained, frozen **CLAP** model → one **embedding** (512 numbers) per window.
3. Give each window a **label**: the average of DEAM's per-second (dynamic) valence and arousal annotations inside that window.
4. Save embeddings + labels to Google Drive, so this slow step never has to run again.

Notebook 02 then trains the baseline, BiLSTM and BiLSTM+attention on this saved data.

**Before running:** `Runtime → Change runtime type → T4 GPU`. Put `DEAM_audio.zip` and `DEAM_Annotations.zip` in `My Drive/C4/DEAM/`.

## 1. Connect Google Drive
Colab's own disk is wiped when the session ends. Google Drive is permanent, so inputs and outputs live there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/C4'
DEAM_ZIPS = f'{DRIVE_DIR}/DEAM'           # where you uploaded the two zip files
OUT_DIR = f'{DRIVE_DIR}/embeddings'       # results of this notebook go here
LOCAL_DIR = '/content/deam'               # unzipped copy on Colab's fast local disk

import os
os.makedirs(OUT_DIR, exist_ok=True)
print(os.listdir(DEAM_ZIPS))

## 2. Settings
These **must match the backend** (`backend/config.py`), otherwise the model is trained on windows that look different from the ones it sees later.

In [ ]:
SAMPLE_RATE = 48000       # CLAP expects 48 kHz audio
SEGMENT_SECONDS = 10      # window length (= CLAP's maximum input length)
HOP_SECONDS = 5           # a new window starts every 5 s
CLAP_MODEL_NAME = 'laion/clap-htsat-unfused'
BATCH_SIZE = 32           # windows sent to the GPU at once

## 3. Unzip DEAM to the local disk
Reading 1,800 small files straight from Drive is very slow, so we copy and unzip once onto Colab's local disk.

In [ ]:
import glob, zipfile

if not os.path.exists(LOCAL_DIR):
    for name in ['DEAM_audio.zip', 'DEAM_Annotations.zip']:
        print('Unzipping', name, '...')
        with zipfile.ZipFile(f'{DEAM_ZIPS}/{name}') as z:
            z.extractall(LOCAL_DIR)

audio_files = glob.glob(f'{LOCAL_DIR}/**/*.mp3', recursive=True)
print(len(audio_files), 'mp3 files found')

## 4. Load the dynamic (per-time-step) annotations
DEAM has two kinds of labels:
- **static**: one rating for the whole song (scale 1–9), and
- **dynamic**: a rating every 0.5 s while listening, starting at 15 s (the first 15 s were left out by the DEAM authors so listeners could settle in).

We use the **dynamic** labels, because we predict emotion **per window**, not per song. Each CSV row is a song; the columns `sample_15000ms`, `sample_15500ms`, … are the time steps.

In [ ]:
import numpy as np
import pandas as pd

def find_csv(kind):
    # Search instead of hard-coding the path, because folder names differ between DEAM versions.
    matches = [p for p in glob.glob(f'{LOCAL_DIR}/**/{kind}.csv', recursive=True) if 'dynamic' in p.lower()]
    assert matches, f'No dynamic {kind}.csv found'
    print(kind, '->', matches[0])
    return pd.read_csv(matches[0])

valence_df = find_csv('valence').set_index('song_id')
arousal_df = find_csv('arousal').set_index('song_id')
print(valence_df.shape, 'songs x time steps')
valence_df.iloc[:3, :6]

### Check the label scale
CLAUDE.md says: if labels are on a 1–9 scale, convert with `(x - 5) / 4` so they fit **[-1, 1]**, like the backend's output. We **check** the real range instead of assuming it.

In [ ]:
vmin, vmax = np.nanmin(valence_df.values), np.nanmax(valence_df.values)
amin, amax = np.nanmin(arousal_df.values), np.nanmax(arousal_df.values)
print(f'valence range: {vmin:.3f} .. {vmax:.3f}')
print(f'arousal range: {amin:.3f} .. {amax:.3f}')

if max(vmax, amax) > 1.0:
    print('Labels look like the 1-9 scale -> converting with (x - 5) / 4')
    valence_df = (valence_df - 5) / 4
    arousal_df = (arousal_df - 5) / 4
else:
    print('Labels are already in [-1, 1] -> no conversion needed')
LABEL_SCALE_NOTE = f'raw valence {vmin:.2f}..{vmax:.2f}, arousal {amin:.2f}..{amax:.2f}'

## 5. Windowing: same 10 s / 5 s hop as the backend
Only the **annotated** region is used, because windows without labels can't teach the model anything.
A window's label = the **mean** of the annotation values whose time falls inside the window.

Example for a 45 s clip annotated from 15 s to 45 s → windows 15–25, 20–30, 25–35, 30–40, 35–45 (5 windows).

In [ ]:
def annotation_times(df):
    # 'sample_15000ms' -> 15.0 seconds
    return np.array([int(c.replace('sample_', '').replace('ms', '')) / 1000 for c in df.columns])

V_TIMES = annotation_times(valence_df)
A_TIMES = annotation_times(arousal_df)

def make_windows(song_id):
    # Returns a list of (start_s, mean_valence, mean_arousal) for one song.
    v = valence_df.loc[song_id].values
    a = arousal_df.loc[song_id].values
    v_t, a_t = V_TIMES[~np.isnan(v)], A_TIMES[~np.isnan(a)]
    v, a = v[~np.isnan(v)], a[~np.isnan(a)]
    if len(v_t) == 0 or len(a_t) == 0:
        return []
    first = max(v_t[0], a_t[0])
    last = min(v_t[-1], a_t[-1]) + 0.5        # each sample covers 0.5 s
    windows = []
    start = first
    while start + SEGMENT_SECONDS <= last + 1e-6:
        end = start + SEGMENT_SECONDS
        v_in = v[(v_t >= start) & (v_t < end)]
        a_in = a[(a_t >= start) & (a_t < end)]
        if len(v_in) and len(a_in):
            windows.append((float(start), float(v_in.mean()), float(a_in.mean())))
        start += HOP_SECONDS
    return windows

example_id = valence_df.index[0]
print('song', example_id, '->', make_windows(example_id))

## 6. Load CLAP (pretrained, frozen)
CLAP was trained by LAION to match audio with text descriptions. Its audio encoder already "understands" sound (instruments, tempo, mood). We **don't change its weights** (`eval()` + `no_grad`); we only use its 512-number output as input features for my own small emotion head.

In [ ]:
import torch
from transformers import ClapModel, ClapProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
processor = ClapProcessor.from_pretrained(CLAP_MODEL_NAME)
clap = ClapModel.from_pretrained(CLAP_MODEL_NAME).to(device).eval()

def embed_batch(audio_windows):
    inputs = processor.feature_extractor(audio_windows, sampling_rate=SAMPLE_RATE, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = clap.get_audio_features(**inputs)
    out = out if torch.is_tensor(out) else out.pooler_output   # transformers 5 wraps the output
    return out.cpu().numpy()

test = embed_batch([np.zeros(SAMPLE_RATE * SEGMENT_SECONDS, dtype=np.float32)])
print('embedding shape for one window:', test.shape)   # expected (1, 512)

## 7. Compute embeddings for every song (with checkpoints)
This is the slow step (roughly 20–40 min on a T4). Colab can disconnect, so results are saved to Drive **every 200 songs**. If the session dies, just run all cells again: finished parts are skipped.

In [ ]:
import librosa
from tqdm.auto import tqdm

audio_by_id = {int(os.path.splitext(os.path.basename(p))[0]): p for p in audio_files}
song_ids = [int(s) for s in valence_df.index if int(s) in audio_by_id and int(s) in arousal_df.index]
print(len(song_ids), 'songs have both audio and dynamic annotations')

PART_SIZE = 200
WINDOW_LEN = SAMPLE_RATE * SEGMENT_SECONDS

def process_part(ids):
    emb, val, aro, sid, starts = [], [], [], [], []
    batch_audio, batch_meta = [], []

    def flush():
        if batch_audio:
            emb.append(embed_batch(batch_audio))
            for (s, st, v, a) in batch_meta:
                sid.append(s); starts.append(st); val.append(v); aro.append(a)
            batch_audio.clear(); batch_meta.clear()

    for song_id in tqdm(ids, leave=False):
        windows = make_windows(song_id)
        if not windows:
            continue
        audio, _ = librosa.load(audio_by_id[song_id], sr=SAMPLE_RATE, mono=True)
        for start, v, a in windows:
            chunk = audio[int(start * SAMPLE_RATE): int(start * SAMPLE_RATE) + WINDOW_LEN]
            if len(chunk) < WINDOW_LEN:
                chunk = np.pad(chunk, (0, WINDOW_LEN - len(chunk)))   # silence padding, same as backend
            batch_audio.append(chunk.astype(np.float32))
            batch_meta.append((song_id, start, v, a))
            if len(batch_audio) == BATCH_SIZE:
                flush()
    flush()
    return dict(embeddings=np.concatenate(emb).astype(np.float32), valence=np.array(val, np.float32),
                arousal=np.array(aro, np.float32), song_id=np.array(sid, np.int32), start_s=np.array(starts, np.float32))

parts_dir = f'{OUT_DIR}/parts'
os.makedirs(parts_dir, exist_ok=True)
for p, i in enumerate(range(0, len(song_ids), PART_SIZE)):
    part_path = f'{parts_dir}/part_{p:02d}.npz'
    if os.path.exists(part_path):
        print('skip (already done):', part_path)
        continue
    np.savez(part_path, **process_part(song_ids[i:i + PART_SIZE]))
    print('saved', part_path)

## 8. Merge the parts into one file
Final file: `deam_clap_windows.npz` with one row per window:

| array | shape | meaning |
|---|---|---|
| `embeddings` | (N, 512) | CLAP embedding of the window |
| `valence`, `arousal` | (N,) | mean dynamic label in [-1, 1] |
| `song_id` | (N,) | which song (needed to split train/val/test **by song** in notebook 02) |
| `start_s` | (N,) | window start time (gives the window order inside a song) |

In [ ]:
import json, datetime, transformers

parts = [np.load(p) for p in sorted(glob.glob(f'{parts_dir}/part_*.npz'))]
data = {k: np.concatenate([p[k] for p in parts]) for k in parts[0].files}
np.savez(f'{OUT_DIR}/deam_clap_windows.npz', **data)

meta = dict(
    created=datetime.datetime.now().isoformat(timespec='seconds'),
    clap_model=CLAP_MODEL_NAME, transformers_version=transformers.__version__,
    sample_rate=SAMPLE_RATE, segment_seconds=SEGMENT_SECONDS, hop_seconds=HOP_SECONDS,
    n_windows=int(len(data['valence'])), n_songs=int(len(np.unique(data['song_id']))),
    label_scale=LABEL_SCALE_NOTE,
)
with open(f'{OUT_DIR}/deam_clap_windows_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))

## 9. Sanity checks + figures for the slides
Quick look at the data before training: how labels are spread, and how many windows each song has (notebook 02's BiLSTM reads each song as a sequence of windows).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(data['valence'], data['arousal'], s=2, alpha=0.3)
axes[0].axhline(0, color='gray'); axes[0].axvline(0, color='gray')
axes[0].set(xlabel='valence', ylabel='arousal', title='DEAM windows on the valence-arousal plane', xlim=(-1, 1), ylim=(-1, 1))
axes[1].hist([data['valence'], data['arousal']], bins=40, label=['valence', 'arousal'])
axes[1].legend(); axes[1].set_title('Label distribution')
_, counts = np.unique(data['song_id'], return_counts=True)
axes[2].hist(counts, bins=range(1, counts.max() + 2))
axes[2].set(xlabel='windows per song', ylabel='songs', title='Sequence length per song')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/deam_data_overview.png', dpi=150)
plt.show()

print('embeddings:', data['embeddings'].shape, '| any NaN:', np.isnan(data['embeddings']).any())
print('windows per song: min', counts.min(), 'median', int(np.median(counts)), 'max', counts.max())